In [10]:

import pickle
import anndata as ad
from cellarium.ml.data import read_h5ad_file

In [1]:
def mask_data(adata, masking_probability, random_seed):
    """
    Randomly mask a fraction of genes per cell in an AnnData object.

    Parameters
    ----------
    adata : AnnData
        Input AnnData object (n_obs x n_vars).
    mask_fraction : float
        Fraction of genes to mask per cell (e.g., 0.2 = 20%).
    mask_value : float or np.nan
        Value to assign to masked positions (default 0.0).
    layer_key : str
        Key in adata.layers where masked data will be stored.
    seed : int or None
        Random seed for reproducibility.

    Returns
    -------
    masked_adata : AnnData
        New AnnData object with:
          - masked counts in `.X` and `.layers[layer_key]`
          - mask matrix in `.layers[f"mask_{int(masking_probability*100)}"]`
          - per-cell mask indices in `.obs[f"masked_gene_indices_{int(masking_probability*100)}"]`
    """

    rng = np.random.default_rng(random_seed)
    
    n_cells, n_genes =  adata.X.shape
    n_mask = int(masking_probability * n_genes)
    print(f"Masking {n_mask} genes out of {n_genes} ({masking_probability*100}%) per cell.")
    
    # Create mask matrix of all zeros of shape (n_cells, n_genes)
    mask = np.zeros(adata.X.shape, dtype=bool)
  
    masked_indices_per_cell = []
    # For each cell, randomly select n_mask gene indices to mask
    for i in range(n_cells):
        # print(f"Processing cell {i+1}/{n_cells}")
        # Randomly choose n_mask unique gene indices
        mask_indices = rng.choice(n_genes, n_mask, replace=False)
        mask[i, mask_indices] = True
        masked_indices_per_cell.append(mask_indices)

    X_masked = adata.X.copy()
    # Apply the mask: set masked positions to 0
    X_masked[mask] = 0

    # Create new AnnData to store masked data and mask matrix
    masked_adata = ad.AnnData(X_masked, obs=adata.obs.copy(), var=adata.var.copy())
    masked_adata.layers["X_masked"] = X_masked
    masked_adata.layers[f"mask_{int(masking_probability*100)}"] = mask.astype(np.int8)
    masked_adata.obsm = adata.obsm.copy()

    # Store which gene indices were masked per cell (as strings for convenience)
    masked_adata.obs[f"masked_gene_indices_{int(masking_probability*100)}"] = [
        ",".join(map(str, idxs)) for idxs in masked_indices_per_cell
    ]

    # Create DataModule for masked data
    dm_mask= CellariumAnnDataDataModule(
            dadc=masked_adata,
            batch_keys={
                "x_ng": AnnDataField(attr="X", convert_fn=densify),
                "var_names_g": AnnDataField(attr="var_names"),
                "obs_names_n": AnnDataField(attr="obs_names"),
                "batch_index_n": AnnDataField(attr="obs", key="batch", convert_fn=categories_to_codes),
            },
            batch_size=512,
            shuffle=False,
        )  

    return masked_adata, dm_mask, masked_indices_per_cell

In [ ]:
# we want to test out the anndata object to show tha timputed values are better than 0
# to do this, we will run test data unchanged through cas
# CAS returns the annotations per cell -- takes an h5ad file as input

# cas takes h5ad file as input


# Read in original data
data_path = "/Users/aseelawdeh/Documents/stephen_scvi/pbmc_count.h5ad"
org_adata = read_h5ad_file(data_path)
print(org_adata)

# Take subset of 1000 cells for testing - randomly select 1000 cells
# set random seed for reproducibility
import numpy as np
np.random.seed(42)
random_indices = np.random.choice(org_adata.n_obs, size=1000, replace=False)

# We will be our baseline dataset -- to put through CAS without any masking
subset_adata = org_adata[random_indices, :]
print(subset_adata)

# Save subset to h5ad file
subset_h5ad_path = "cas_inputs/baseline_subset.h5ad"
subset_adata.write_h5ad(subset_h5ad_path)

AnnData object with n_obs × n_vars = 31774 × 4000
    obs: 'batch', 'chemistry', 'data_type', 'dpt_pseudotime', 'final_annotation', 'mt_frac', 'n_counts', 'n_genes', 'sample_ID', 'size_factors', 'species', 'study', 'tissue', 'condition', 'concat'
View of AnnData object with n_obs × n_vars = 1000 × 4000
    obs: 'batch', 'chemistry', 'data_type', 'dpt_pseudotime', 'final_annotation', 'mt_frac', 'n_counts', 'n_genes', 'sample_ID', 'size_factors', 'species', 'study', 'tissue', 'condition', 'concat'


In [45]:
#data_path = "runs/imputation_0.99/mask_50/imputation_reconstructed_whole.h5ad"
data_path = "runs/imputation_0.99/mask_50/imputation_reconstructed_masked.h5ad"
reconstructed_adata = read_h5ad_file(data_path)

with open(f"runs/imputation_0.99/mask_50/predictions.pkl", "rb") as f:
    predictions = pickle.load(f)
    
gene_idx_list = predictions["mask50_masked_gene_indices"]#[random_indices]
gene_idx_list = [gene_idx_list[i] for i in random_indices]

reconstructed_adata.X = reconstructed_adata.layers["reconstructed_masked"]

# print(reconstructed_adata)
# check if cells in reconstructed matched those in original -- the order matches
print("Checking if cell names match between reconstructed and original data:", reconstructed_adata.obs_names.equals(org_adata.obs_names))

subset_reconstructed_adata = reconstructed_adata[random_indices, :]
print("Save the reconstructed subset h5ad file for subset of 1000 cells")
subset_reconstructed_adata.write_h5ad("cas_inputs/reconstructed_subset.h5ad")

# for the gene indices, set those genes to 0 in the reconstructed_adata 
for i, gene_idx in enumerate(gene_idx_list):
    subset_reconstructed_adata.X[i, gene_idx.tolist()] = 0 

print("Save the zerovalue subset h5ad file for subset of 1000 cells")
subset_reconstructed_adata.write_h5ad("cas_inputs/zerovalue_subset.h5ad")
print(subset_reconstructed_adata)

Checking if cell names match between reconstructed and original data: True
Save the reconstructed subset h5ad file for subset of 1000 cells


/var/folders/vv/z6gg91c17fv4f2g9vzyk4hgh0000gn/T/ipykernel_1176/1810189937.py:23: ImplicitModificationWarning: Trying to modify attribute `.X` of view, initializing view as actual.
  subset_reconstructed_adata.X[i, gene_idx.tolist()] = 0


Save the zerovalue subset h5ad file for subset of 1000 cells
AnnData object with n_obs × n_vars = 1000 × 4000
    obs: 'batch', 'chemistry', 'data_type', 'dpt_pseudotime', 'final_annotation', 'mt_frac', 'n_counts', 'n_genes', 'sample_ID', 'size_factors', 'species', 'study', 'tissue', 'condition', 'concat'
    layers: 'reconstructed_masked'
